# Train Mask2Former Swin-T (Hugging Face transformers)

Thin runner over `iiss.models.m2f.train` (same code as `python -m iiss.train --run configs/runs/m2f_s<S>_seed42.yaml`). One notebook execution = one run; change only the parameters cell to launch another S.

- Run order (priority): the S=640 baseline first, then S=320 and S=960 (`.agent/experiments.md`).
- The 2-hour cap is cumulative per run id over all sessions, including hard kills (logged as `killed`). A capped or stopped run is continued with `RESUME = True`; a fresh start refuses to overwrite an existing `weights/last.pt`.
- The evaluated checkpoint is `weights/last.pt`; `best.pt` is logged only.
- Keep the outputs of this notebook after a real run: they are process evidence.
- `SMOKE = True`: 16 train / 8 valid images, S=320, 2 epochs, 5-minute cap, written to `runs/smoke/`.
- Keep the laptop plugged in and awake; close other GPU programs.

## Parameters

In [ ]:
MODEL = 'm2f'
S = 640              # 320 / 640 / 960 -> configs/runs/m2f_s<S>_seed<SEED>.yaml
SEED = 42
RESUME = False       # True: continue from weights/last.pt (after a stop, crash or cap)
RUNS_ROOT = None     # None = runs/ in the repo (override, e.g. Google Drive on Colab)
COLAB = False        # True: run the Colab setup cell below first
REPO_URL = 'https://github.com/luminolous/indoor-instance-seg-shift.git'
DRIVE_DATASET = '/content/drive/MyDrive/iiss/panoptic-indoor-segmentation.v5i.coco-segmentation'
DRIVE_RUNS = '/content/drive/MyDrive/iiss/runs'
SMOKE = False

## Colab setup (optional)

Only for runs that do not fit locally. Skipped when `COLAB = False`. Not tested by Claude Code (no Colab access): check the paths before relying on it.

Before running: upload the dataset folder `panoptic-indoor-segmentation.v5i.coco-segmentation` to `DRIVE_DATASET` on Google Drive. Runs are written to `DRIVE_RUNS` so they survive a disconnect; resume with `RESUME = True` after reconnecting. Log the Colab GPU in `logbook.md`: speed and memory differ from the laptop.

In [ ]:
if COLAB:
    import os, shutil, subprocess
    from google.colab import drive

    drive.mount('/content/drive')
    if not os.path.isdir('/content/indoor-instance-seg-shift'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/indoor-instance-seg-shift'], check=True)
    os.chdir('/content/indoor-instance-seg-shift')
    subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
    dst = 'dataset/panoptic-indoor-segmentation.v5i.coco-segmentation'
    if not os.path.isdir(dst):
        shutil.copytree(DRIVE_DATASET, dst)
    RUNS_ROOT = DRIVE_RUNS

## Config

In [ ]:
import importlib

import pandas as pd
import torch
from IPython.display import display

from iiss.analysis.training import (config_table, gpu_sessions, notebook_config,
                                    plot_history, run_summary)
from iiss.models.common import run_dir
from iiss.utils.log import setup_logging

setup_logging()
cfg = notebook_config(MODEL, S, SEED, smoke=SMOKE, runs_root=RUNS_ROOT)
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
config_table(cfg)

## Status before training

In [ ]:
pd.Series(run_summary(cfg)).to_frame('value')

## Train

Logs stream below. Interrupting the kernel stops the run safely (state of the last finished epoch is kept).

In [ ]:
module = importlib.import_module(f'iiss.models.{MODEL}')
out = module.train(cfg, resume=RESUME)
out

## Result

GPU time per session (`runs/gpu_time_log.csv`), run status and training curves (also saved as `<run_dir>/curves.png`). The validation curve of YOLO is Ultralytics' validator; the comparable numbers come from `predict_and_evaluate.ipynb`.

In [ ]:
display(gpu_sessions(cfg))
display(pd.Series(run_summary(cfg)).to_frame('value'))
fig = plot_history(cfg, save_to=run_dir(cfg) / 'curves.png')

## Next step

Send back the summary table above (run id, epochs, GPU hours, last/best valid metric). Then run the next S, or `predict_and_evaluate.ipynb`.